In [1]:
!pip install faiss
from sentence_transformers import SentenceTransformer
import pandas as pd
import torch
import faiss
import numpy as np
import openai
import json

ERROR: Could not find a version that satisfies the requirement faiss (from versions: none)
ERROR: No matching distribution found for faiss


ModuleNotFoundError: No module named 'faiss'

In [ ]:
def retrieve_top_k(query, k=5):
    # Encode the query
    query_embedding = model.encode(query, convert_to_tensor=True)

    # Convert query embedding to numpy array
    query_embedding_np = query_embedding.cpu().detach().numpy()

    # Search the index
    distances, indices = index.search(np.array([query_embedding_np]), k)

    # Retrieve the top-k results
    results = data.iloc[indices[0]]
    return results

def generate_answer_chatgpt(conversation_history, model="gpt-4o-mini", max_tokens=500, temperature=0.7):
    # Make a request to the OpenAI API
    response = openai.chat.completions.create(
        model=model,
        messages=conversation_history,
        max_tokens=max_tokens,
        n=1,
        stop=None,
        temperature=temperature
    )

    # Extract the answer from the response
    answer = response.choices[0].message.content.strip()
    return answer

In [ ]:
input_file = 'result.xlsx'
output_file = 'responses.xlsx'

# Load your dataset
data = pd.read_excel(input_file)

In [ ]:
# Load the multilingual Sentence-BERT model
model = SentenceTransformer('distiluse-base-multilingual-cased-v1')

# # Create embeddings for context and question
# context_embeddings = model.encode(data['context'].tolist(), convert_to_tensor=True)
# question_embeddings = model.encode(data['question'].tolist(), convert_to_tensor=True)

# Load the embeddings from files
context_embeddings_np = np.load('context_embeddings.npy')
question_embeddings_np = np.load('question_embeddings.npy')

# Convert numpy arrays back to tensors if needed
context_embeddings = torch.tensor(context_embeddings_np)
question_embeddings = torch.tensor(question_embeddings_np)

# Use only question embeddings for the FAISS index
embeddings_np = question_embeddings.cpu().detach().numpy()

# Create a FAISS index
dimension = embeddings_np.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings_np)

e:\1.apps\python\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


e:\1.apps\python\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
# Set your OpenAI API key
key_path = './config.json'
with open(key_path, "r") as file:
    config = json.load(file)
    openai.api_key = config["openai_api_key"]

In [ ]:
# Initialize conversation history
conversation_history = [
    {"role": "system", "content": "You are a helpful assistant that responds in Vietnamese."}
]

# Example multi-turn conversation
questions = [
    "Chuyện gì xảy ra với Nick khi được sinh ra?",
    "Điều đó đã ảnh hưởng đến cuộc sống hiện tại của anh ấy như thế nào?",
    # "Làm thế nào anh ấy vượt qua khó khăn?"
]

In [ ]:
responses = []
for question in questions:
    # Retrieve top-k results for the current query
    top_k_results = retrieve_top_k(question)

    # Concatenate the contexts from the top-k results
    combined_context = " ".join(top_k_results['context'].tolist())

    # Add user query and context to the conversation history
    conversation_history.append({"role": "user", "content": f"Question: {question}\nContext: {combined_context}\nAnswer:"})

    # Generate answer using ChatGPT
    answer = generate_answer_chatgpt(conversation_history)

    # Add assistant's response to the conversation history
    conversation_history.append({"role": "assistant", "content": answer})

    # Print the answer
    print(f"User: {question}")
    print(f"Assistant: {answer}\n")

    # Add into responses
    responses.append({"question": question, "answer": answer})

User: Chuyện gì xảy ra với Nick khi được sinh ra?
Assistant: Khi Nick Vujicic được sinh ra, anh đã chào đời với tình trạng không có tay và chân. Phản ứng đầu tiên của mẹ anh là rất đau khổ và không thể chấp nhận sự thật này. Mẹ anh đã yêu cầu các nhân viên y tế mang anh đi vì bà không muốn chạm vào hay nhìn thấy đứa con tật nguyền của mình. Cha anh cảm thấy buồn vì mẹ anh không được chuẩn bị tâm lý để đối mặt với tình huống này. Sau đó, khi cha anh đến thăm Nick, ông đã cố gắng khuyến khích mẹ anh gặp Nick, nhưng bà vẫn từ chối. Hành trình của Nick từ khi sinh ra cho đến khi trở thành một người truyền cảm hứng cho nhiều người khác là một câu chuyện đầy cảm xúc và nghị lực.

User: Điều đó đã ảnh hưởng đến cuộc sống hiện tại của anh ấy như thế nào?
Assistant: Cuộc sống hiện tại của Nick Vujicic bị ảnh hưởng sâu sắc bởi những trải nghiệm và thách thức mà anh đã phải đối mặt từ khi sinh ra. Sự thiếu hụt tay chân không chỉ là một trở ngại về thể chất mà còn là một bài học lớn trong việc chấ

In [ ]:
# Convert into DataFrame
df = responses.to_pandas()
df.head()

# Save the DataFrame to an Excel file
df.to_excel(output_file, index=False)